# Pre-trained CNN Baseline Demo

Shows that off-the-shelf ImageNet CNNs **fail** on our tea images — justifying why fine-tuning is needed.

Run on **CPU**, no training required.

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torchvision import models
from torchvision.models import (
    VGG16_Weights, ResNet50_Weights,
    EfficientNet_B3_Weights, MobileNet_V3_Large_Weights
)

ROOT     = Path('..')
DATA_DIR = ROOT / 'tea_dataset'
df       = pd.read_csv(DATA_DIR / 'metadata_clean.csv')

CLASSES = ['cha_dam_yen','cha_khiao_nom','cha_nom_khai_muk','cha_phonlamai','cha_thai']
print('Dataset loaded:', len(df), 'images')

In [ ]:
# ── Load all 4 pre-trained models (no fine-tuning) ────────────────────────────
MODELS = {
    'VGG-16':              (models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1),           VGG16_Weights.IMAGENET1K_V1),
    'ResNet-50':           (models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2),     ResNet50_Weights.IMAGENET1K_V2),
    'EfficientNet-B3':     (models.efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1), EfficientNet_B3_Weights.IMAGENET1K_V1),
    'MobileNet-V3-Large':  (models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2), MobileNet_V3_Large_Weights.IMAGENET1K_V2),
}
for name, (m, _) in MODELS.items():
    m.eval()
    print(f'Loaded {name}')

In [ ]:
# ── Predict function ──────────────────────────────────────────────────────────
@torch.no_grad()
def predict_top3(model, weights, img_path):
    img        = Image.open(img_path).convert('RGB')
    tensor     = weights.transforms()(img).unsqueeze(0)
    probs      = model(tensor).softmax(1)
    top3_probs, top3_ids = probs.topk(3, dim=1)
    meta       = weights.meta
    categories = meta['categories']
    return [(categories[i], round(p.item()*100, 1))
            for i, p in zip(top3_ids[0], top3_probs[0])]

In [ ]:
# ── Show baseline predictions for 1 sample per class ─────────────────────────
# Pick 1 image per class
samples = {cls: DATA_DIR / df[df['class']==cls].iloc[0]['relpath'] for cls in CLASSES}

arch_name = 'ResNet-50'   # change to see other models
model, weights = MODELS[arch_name]

fig, axes = plt.subplots(1, len(CLASSES), figsize=(18, 5))
for ax, cls in zip(axes, CLASSES):
    img  = Image.open(samples[cls]).convert('RGB')
    top3 = predict_top3(model, weights, samples[cls])
    img.thumbnail((200, 200))
    ax.imshow(img)
    ax.axis('off')
    label = f"GT: {cls.replace('cha_','')}\n"
    for name, pct in top3:
        label += f"{name[:20]}: {pct}%\n"
    ax.set_title(label, fontsize=7.5, color='red')

plt.suptitle(f'Pre-trained {arch_name} (ImageNet only) — Before Fine-tuning\nPrediction results on our dataset',
             fontsize=11)
plt.tight_layout()
plt.savefig(f'baseline_{arch_name.lower().replace("-","_")}.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNote: None of the predictions match our tea classes — this justifies fine-tuning.')

In [ ]:
# ── Repeat for all 4 architectures ───────────────────────────────────────────
for arch_name, (model, weights) in MODELS.items():
    fig, axes = plt.subplots(1, len(CLASSES), figsize=(18, 4))
    for ax, cls in zip(axes, CLASSES):
        img  = Image.open(samples[cls]).convert('RGB')
        top3 = predict_top3(model, weights, samples[cls])
        img.thumbnail((180, 180))
        ax.imshow(img)
        ax.axis('off')
        label = f"GT: {cls.replace('cha_','')}\n"
        for name, pct in top3:
            label += f"{name[:18]}: {pct}%\n"
        ax.set_title(label, fontsize=7, color='red')
    plt.suptitle(f'Pre-trained {arch_name} (ImageNet) — Before Fine-tuning', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'baseline_{arch_name.lower().replace(" ","_").replace("-","_")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()